# FOXF1_bead — 06_live_plotting

**Feeds:** Fig 2c, ED Fig 3d

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 06 Live Plotting

## Dependency Note

This notebook loads live preprocessing outputs written by `05_live_quantification.ipynb`, including the transferred DAPI masks, global flat-field estimate, and per-image off-mask background fits.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print("ROOT:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from scripts import pipeline_common as common
from scripts import live_fixed_quantification as lfq
from scripts import live_fixed_alignment as lfa

## Paths

In [ ]:
POSITION_MANIFEST = ROOT / "results/manifests/analysis_position_manifest.tsv"
LIVE_SMALL_CENTROIDS = ROOT / "results/annotations/well_centroids_small_mapped.tsv"

OUT_DIR = ROOT / "results/measurements/live_fixed_small"
FIGURE_DIR_ROOT = ROOT / "results/figures"
FIG_DIR = FIGURE_DIR_ROOT / "06"
CANDIDATE_FIGURE_DIR = FIG_DIR / "candidate_figures"
ALTERNATE_FIGURE_DIR = FIG_DIR / "alternate_figures"
MASK_DIR = ROOT / "results/masks/live_fixed_small"
LIVE_MASK_FINAL_DIR = MASK_DIR / "live_mask_final04_npz"
LIVE_FIELD_NPZ = OUT_DIR / "live_yfp_flatfield_field.npz"
LIVE_FIELD_SUMMARY_TSV = OUT_DIR / "live_yfp_flatfield_summary.tsv"
LIVE_MASK_SUMMARY_TSV = OUT_DIR / "live_mask_transfer_summary.tsv"
PER_IMAGE_BG_TSV = OUT_DIR / "live_yfp_per_image_background_params.tsv"
for path in [OUT_DIR, FIG_DIR, CANDIDATE_FIGURE_DIR, ALTERNATE_FIGURE_DIR, MASK_DIR, LIVE_MASK_FINAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LIVE_COHORT_ID = "2026-01-22_day2_live"
LIVE_YFP_KEYWORDS = ["tagyfp", "foxf1", "yfp"]
BIN_UM = 35.0

pos_df = common.load_position_manifest(POSITION_MANIFEST)
live_pos_df = common.filter_position_manifest(
    pos_df=pos_df,
    cohort_ids=[LIVE_COHORT_ID],
    conditions=["live"],
).sort_values(["canonical_position"]).reset_index(drop=True)
live_pos_df["canonical_position"] = live_pos_df["canonical_position"].astype(str)
live_by_pos = {str(r.canonical_position): pd.Series(r._asdict()) for r in live_pos_df.itertuples(index=False)}

live_cent_df = pd.read_csv(LIVE_SMALL_CENTROIDS, sep="	")
live_cent_df["canonical_position"] = live_cent_df["canonical_position"].astype(str)


def load_live_bundle(canonical_position: str) -> dict:
    row = live_by_pos[str(canonical_position)]
    img = common.read_czi(ROOT / row["primary_analysis_file"])
    bf_idx = common.find_channel_index(img.channels, ["bright"])
    yfp_idx = common.find_channel_index(img.channels, LIVE_YFP_KEYWORDS)
    if bf_idx is None or yfp_idx is None:
        raise RuntimeError(f"Missing live BF or FOXF1-YFP reporter channel for {canonical_position}; channels={img.channels}")
    return {
        "row": row,
        "image": img,
        "bf": img.channel_images[int(bf_idx)].astype(np.float32),
        "yfp_raw": img.channel_images[int(yfp_idx)].astype(np.float32),
    }


## Load Live Quantification Intermediates

In [ ]:
required_outputs = [
    LIVE_MASK_SUMMARY_TSV,
    PER_IMAGE_BG_TSV,
    LIVE_FIELD_NPZ,
    LIVE_FIELD_SUMMARY_TSV,
]
missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Run 05 Live Quantification first. Missing outputs:\n" + "\n".join(missing)
    )

live_mask_summary_df = pd.read_csv(LIVE_MASK_SUMMARY_TSV, sep="	")
live_mask_summary_df["canonical_position"] = live_mask_summary_df["canonical_position"].astype(str)
per_image_bg_df = pd.read_csv(PER_IMAGE_BG_TSV, sep="	")
per_image_bg_df["canonical_position"] = per_image_bg_df["canonical_position"].astype(str)
live_field_summary_df = pd.read_csv(LIVE_FIELD_SUMMARY_TSV, sep="	")

with np.load(LIVE_FIELD_NPZ, allow_pickle=True) as payload:
    live_field = np.asarray(payload["field"], dtype=np.float32)

ok_positions = live_mask_summary_df.loc[live_mask_summary_df["status"] == "ok", "canonical_position"].astype(str).tolist()
live_masks_by_position = {}
missing_masks = []
for pos in ok_positions:
    path = LIVE_MASK_FINAL_DIR / f"{pos}_live_mask_from_fixed_final04.npz"
    if not path.exists():
        missing_masks.append(str(path))
        continue
    with np.load(path, allow_pickle=True) as payload:
        live_masks_by_position[str(pos)] = np.asarray(payload["mask"], dtype=bool)

if missing_masks:
    raise FileNotFoundError(
        "Missing transferred live masks written by 05:\n" + "\n".join(missing_masks[:10])
    )

ok_positions = [p for p in ok_positions if p in live_masks_by_position]

print("Loaded live quantification intermediates from 05:")
print(f"ok positions: {len(ok_positions)}")
print(f"live field shape: {tuple(live_field.shape)}")
display(live_mask_summary_df.head())
print()
display(per_image_bg_df.head())
print()
display(live_field_summary_df)


## Live FOXF1-YFP Reporter Activity Inside Transferred DAPI Masks

In [ ]:
def aggregate_trace_across_images(
    trace_df: pd.DataFrame,
    x_col: str = "bin_mid_um",
    y_col: str = "mean_value",
    unit_col: str = "canonical_position",
) -> pd.DataFrame:
    if len(trace_df) == 0:
        return pd.DataFrame(columns=["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", x_col, "mean", "sd", "sem", "n_images"])
    df = trace_df[np.isfinite(trace_df[x_col]) & np.isfinite(trace_df[y_col])].copy()
    if len(df) == 0:
        return pd.DataFrame(columns=["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", x_col, "mean", "sd", "sem", "n_images"])
    rows = []
    group_cols = ["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", x_col]
    for keys, sub in df.groupby(group_cols, sort=True):
        vals = sub[y_col].to_numpy(dtype=float)
        n = int(sub[unit_col].astype(str).nunique())
        sd = float(np.nanstd(vals, ddof=1)) if n > 1 else 0.0
        sem = float(sd / np.sqrt(n)) if n > 1 else 0.0
        rows.append(
            {
                "measurement_name": str(keys[0]),
                "bin_idx": int(keys[1]),
                "bin_start_um": float(keys[2]),
                "bin_end_um": float(keys[3]),
                x_col: float(keys[4]),
                "mean": float(np.nanmean(vals)),
                "sd": sd,
                "sem": sem,
                "n_images": n,
            }
        )
    return pd.DataFrame(rows).sort_values(["measurement_name", x_col]).reset_index(drop=True)


def weighted_trace_pixels(df: pd.DataFrame) -> pd.DataFrame:
    if len(df) == 0:
        return pd.DataFrame(columns=[
            "measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um",
            "total_count_px", "total_weighted_sum", "weighted_mean_value", "pooled_sd_value", "pooled_sem_value"
        ])
    tmp = df.copy()
    counts = tmp["count_px"].astype(float)
    means = tmp["mean_value"].astype(float)
    stds = pd.to_numeric(tmp["std_value"], errors="coerce").astype(float)
    within_ss = np.where((counts > 1) & np.isfinite(stds), np.square(stds) * np.maximum(counts - 1.0, 0.0), 0.0)
    tmp["weighted_sum"] = means * counts
    tmp["sum_x2"] = within_ss + counts * np.square(means)
    agg = (
        tmp.groupby(["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um"], as_index=False)
        .agg(
            total_count_px=("count_px", "sum"),
            total_weighted_sum=("weighted_sum", "sum"),
            total_sum_x2=("sum_x2", "sum"),
        )
    )
    total_n = np.maximum(agg["total_count_px"].astype(float), 1.0)
    agg["weighted_mean_value"] = agg["total_weighted_sum"] / total_n
    numer = agg["total_sum_x2"] - np.square(agg["total_weighted_sum"]) / total_n
    denom = np.maximum(total_n - 1.0, 1.0)
    agg["pooled_sd_value"] = np.sqrt(np.maximum(numer / denom, 0.0))
    agg.loc[agg["total_count_px"].astype(float) <= 1.0, "pooled_sd_value"] = 0.0
    agg["pooled_sem_value"] = agg["pooled_sd_value"] / np.sqrt(total_n)
    return agg.sort_values(["measurement_name", "bin_mid_um"]).reset_index(drop=True)


def collect_live_masked_pixels_by_position() -> dict[str, dict[str, np.ndarray]]:
    payloads: dict[str, dict[str, np.ndarray]] = {}
    for pos in ok_positions:
        live_ctx = load_live_bundle(pos)
        live_mask = np.asarray(live_masks_by_position[pos], dtype=bool)
        corrected = lfq.apply_illumination_field(live_ctx["yfp_raw"], live_field)
        mu_i = float(per_image_mu_map[str(pos)])
        bgsub = common.subtract_uniform_background(
            image=corrected,
            mu_bg_raw=mu_i,
            clip_below_zero=True,
        )
        cent_sub = live_cent_df[
            (live_cent_df["canonical_position"].astype(str) == str(pos))
            & (live_cent_df["mapping_status"] == "ok")
            & (live_cent_df["annotation_status"] == "annotated")
        ].copy()
        beads_xy = cent_sub[["centroid_small_x_px", "centroid_small_y_px"]].to_numpy(dtype=np.float32)
        if beads_xy.size == 0:
            continue
        dist_um = common.nearest_bead_distance_map_um(
            image_shape_yx=bgsub.shape,
            centroid_xy_px=beads_xy,
            pixel_um_x=float(live_ctx["image"].pixel_um_x),
            pixel_um_y=float(live_ctx["image"].pixel_um_y),
        )
        keep = live_mask & np.isfinite(bgsub) & np.isfinite(dist_um)
        if not np.any(keep):
            continue
        payloads[str(pos)] = {
            "distance_um": dist_um[keep].astype(np.float32),
            "value": bgsub[keep].astype(np.float32),
        }
    return payloads


def concatenate_pixel_payloads(payloads: dict[str, dict[str, np.ndarray]]) -> tuple[np.ndarray, np.ndarray]:
    if not payloads:
        return np.array([], dtype=np.float32), np.array([], dtype=np.float32)
    dists = [np.asarray(v["distance_um"], dtype=np.float32) for v in payloads.values() if len(v.get("distance_um", []))]
    vals = [np.asarray(v["value"], dtype=np.float32) for v in payloads.values() if len(v.get("value", []))]
    if not dists:
        return np.array([], dtype=np.float32), np.array([], dtype=np.float32)
    return np.concatenate(dists), np.concatenate(vals)


def equal_support_trace(distance_um: np.ndarray, values: np.ndarray, pixels_per_bin: int, measurement_name: str) -> pd.DataFrame:
    dist = np.asarray(distance_um, dtype=np.float32)
    vals = np.asarray(values, dtype=np.float32)
    keep = np.isfinite(dist) & np.isfinite(vals)
    dist = dist[keep]
    vals = vals[keep]
    if dist.size == 0:
        return pd.DataFrame(columns=["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um", "count_px", "mean_value", "median_value", "std_value", "sem_value"])
    order = np.argsort(dist, kind="mergesort")
    dist = dist[order]
    vals = vals[order]
    rows = []
    pixels_per_bin = max(1, int(pixels_per_bin))
    for start in range(0, len(dist), pixels_per_bin):
        stop = min(len(dist), start + pixels_per_bin)
        d = dist[start:stop]
        v = vals[start:stop]
        n = int(len(v))
        std = float(np.nanstd(v, ddof=1)) if n > 1 else 0.0
        sem = float(std / np.sqrt(n)) if n > 1 else 0.0
        rows.append(
            {
                "measurement_name": str(measurement_name),
                "bin_idx": int(len(rows)),
                "bin_start_um": float(np.nanmin(d)),
                "bin_end_um": float(np.nanmax(d)),
                "bin_mid_um": float(np.nanmean(d)),
                "count_px": n,
                "mean_value": float(np.nanmean(v)),
                "median_value": float(np.nanmedian(v)),
                "std_value": std,
                "sem_value": sem,
            }
        )
    return pd.DataFrame(rows)


def aggregate_equal_support_across_images(
    payloads: dict[str, dict[str, np.ndarray]],
    equal_support_df: pd.DataFrame,
    measurement_name: str,
) -> pd.DataFrame:
    rows = []
    for _, rr in equal_support_df.sort_values("bin_idx").iterrows():
        lo = float(rr["bin_start_um"])
        hi = float(rr["bin_end_um"])
        image_means = []
        image_counts = []
        for pos, payload in payloads.items():
            dist = np.asarray(payload["distance_um"], dtype=np.float32)
            vals = np.asarray(payload["value"], dtype=np.float32)
            keep = np.isfinite(dist) & np.isfinite(vals) & (dist >= lo - 1e-9) & (dist <= hi + 1e-9)
            if not np.any(keep):
                continue
            vv = vals[keep]
            image_means.append(float(np.nanmean(vv)))
            image_counts.append(int(vv.size))
        arr = np.asarray(image_means, dtype=float)
        n = int(np.sum(np.isfinite(arr)))
        mean = float(np.nanmean(arr)) if n else np.nan
        sd = float(np.nanstd(arr, ddof=1)) if n > 1 else (0.0 if n == 1 else np.nan)
        sem = float(sd / np.sqrt(n)) if n > 1 else (0.0 if n == 1 else np.nan)
        rows.append(
            {
                "measurement_name": str(measurement_name),
                "bin_idx": int(rr["bin_idx"]),
                "bin_start_um": lo,
                "bin_end_um": hi,
                "bin_mid_um": float(rr["bin_mid_um"]),
                "bin_width_um": float(hi - lo),
                "total_count_px": int(np.sum(image_counts)) if image_counts else 0,
                "n_images": n,
                "mean": mean,
                "sd": sd,
                "sem": sem,
            }
        )
    return pd.DataFrame(rows).sort_values("bin_idx").reset_index(drop=True)


def _pretty_live_measurement_name(name: str) -> str:
    return {"live_tagyfp_flatfield_per_image_bgsub": "Live FOXF1-YFP reporter activity (flat-field corrected, per-image bg-subtracted)"}.get(str(name), str(name))


def _plot_live_distance_trace(
    ax,
    per_image_df: pd.DataFrame,
    agg_df: pd.DataFrame,
    x_col: str,
    y_col: str,
    band_col: str,
    title: str,
    color: str = "#2e8b57",
    band_label: str = "Mean ± SD across images",
    ylabel: str = "Normalized mean FOXF1-YFP reporter activity (live)\n(flat-field corrected, per-image bg-subtracted; aggregate peak bin = 1)",
):
    first_gray = True
    for _, sub in per_image_df.groupby("canonical_position", sort=True):
        sub = sub.sort_values("bin_mid_um")
        ax.plot(
            sub["bin_mid_um"],
            sub["mean_value"],
            color="0.65",
            alpha=0.14,
            linewidth=0.8,
            label="Per-image traces" if first_gray else None,
        )
        ax.scatter(
            sub["bin_mid_um"],
            sub["mean_value"],
            color="0.65",
            alpha=0.09,
            s=6,
        )
        first_gray = False
    if len(agg_df) > 0:
        agg_df = agg_df.sort_values(x_col)
        mean_vals = agg_df[y_col].to_numpy(dtype=float)
        band_vals = agg_df[band_col].to_numpy(dtype=float)
        ax.plot(
            agg_df[x_col],
            mean_vals,
            color=color,
            linewidth=2.6,
            label="Live FOXF1-YFP reporter activity",
        )
        ax.fill_between(
            agg_df[x_col],
            mean_vals - band_vals,
            mean_vals + band_vals,
            color=color,
            alpha=0.18,
            label=band_label,
        )
    ax.set_xlabel("Distance from nearest bead (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best", frameon=False)


per_image_mu_map = per_image_bg_df.set_index("canonical_position")["mu_bg_raw"].astype(float).to_dict()


live_rows = []
for pos in ok_positions:
    live_ctx = load_live_bundle(pos)
    live_mask = np.asarray(live_masks_by_position[pos], dtype=bool)
    corrected = lfq.apply_illumination_field(live_ctx["yfp_raw"], live_field)
    mu_i = float(per_image_mu_map[str(pos)])
    bgsub = common.subtract_uniform_background(
        image=corrected,
        mu_bg_raw=mu_i,
        clip_below_zero=True,
    )
    cent_sub = live_cent_df[
        (live_cent_df["canonical_position"].astype(str) == str(pos))
        & (live_cent_df["mapping_status"] == "ok")
        & (live_cent_df["annotation_status"] == "annotated")
    ].copy()
    beads_xy = cent_sub[["centroid_small_x_px", "centroid_small_y_px"]].to_numpy(dtype=np.float32)
    if beads_xy.size == 0:
        continue
    dist_um = common.nearest_bead_distance_map_um(
        image_shape_yx=bgsub.shape,
        centroid_xy_px=beads_xy,
        pixel_um_x=float(live_ctx["image"].pixel_um_x),
        pixel_um_y=float(live_ctx["image"].pixel_um_y),
    )
    stats = lfq.masked_distance_bin_stats(
        value_img=bgsub,
        distance_um_map=dist_um,
        mask=live_mask,
        bin_um=float(BIN_UM),
    )
    for _, tr in stats.iterrows():
        live_rows.append(
            {
                "canonical_position": pos,
                "image_id": str(live_ctx["row"].get("image_id", f"{LIVE_COHORT_ID}_{pos}")),
                "cohort_id": LIVE_COHORT_ID,
                "small_file_path": str(live_ctx["row"]["primary_analysis_file"]),
                "measurement_name": "live_tagyfp_flatfield_per_image_bgsub",
                "mask_source": "approved_final04_transferred_dapi",
                "bead_distance_mode": "nearest",
                "bin_idx": int(tr["bin_idx"]),
                "bin_start_um": float(tr["bin_start_um"]),
                "bin_end_um": float(tr["bin_end_um"]),
                "bin_mid_um": float(tr["bin_mid_um"]),
                "count_px": int(tr["count_px"]),
                "mean_value": float(tr["mean_value"]),
                "median_value": float(tr["median_value"]),
                "std_value": float(tr["std_value"]) if pd.notna(tr["std_value"]) else np.nan,
                "sem_value": float(tr["sem_value"]) if pd.notna(tr["sem_value"]) else np.nan,
            }
        )

live_stats_df = pd.DataFrame(live_rows).sort_values(["measurement_name", "canonical_position", "bin_idx"]).reset_index(drop=True)
live_stats_df.to_csv(OUT_DIR / "live_foxf1_pixel_bin_stats.tsv", sep="\t", index=False)

live_trace_df = aggregate_trace_across_images(live_stats_df)
live_trace_df.to_csv(OUT_DIR / "live_tagyfp_distance_trace_across_images.tsv", sep="\t", index=False)

live_trace_merged_df = weighted_trace_pixels(live_stats_df)
live_trace_merged_df.to_csv(OUT_DIR / "live_tagyfp_distance_trace_all_pixels_merged.tsv", sep="\t", index=False)

pixel_payloads = collect_live_masked_pixels_by_position()
live_dist_all_um, live_value_all = concatenate_pixel_payloads(pixel_payloads)
if len(live_trace_merged_df) == 0:
    equal_support_target_px = 1
else:
    equal_support_target_px = int(np.nanmedian(live_trace_merged_df["total_count_px"].astype(float)))
live_equal_support_df = equal_support_trace(
    live_dist_all_um,
    live_value_all,
    pixels_per_bin=equal_support_target_px,
    measurement_name="live_tagyfp_flatfield_per_image_bgsub_equal_support",
)
live_equal_support_df.to_csv(OUT_DIR / "live_tagyfp_distance_trace_equal_support.tsv", sep="\t", index=False)

live_equal_support_across_images_df = aggregate_equal_support_across_images(
    pixel_payloads,
    live_equal_support_df,
    measurement_name="live_tagyfp_flatfield_per_image_bgsub_equal_support_across_images",
)
live_equal_support_across_images_df.to_csv(
    OUT_DIR / "live_tagyfp_distance_trace_equal_support_across_images.tsv",
    sep="\t",
    index=False,
)

summary_txt = (
    f"positions_ok\t{len(ok_positions)}\n"
    f"live_bin_rows\t{len(live_stats_df)}\n"
    f"illumination_field_p01\t{float(live_field_summary_df['field_p01'].iloc[0]):.6f}\n"
    f"illumination_field_p99\t{float(live_field_summary_df['field_p99'].iloc[0]):.6f}\n"
    f"equal_support_target_pixels\t{int(equal_support_target_px)}\n"
    f"measurement_strategy\tflatfield_correct_then_per_image_offmask_bgsubtract_then_measure_inside_transferred_dapi_mask\n"
)
(OUT_DIR / "live_measurement_summary.txt").write_text(summary_txt)

main_agg_raw = live_trace_df.sort_values("bin_mid_um").copy()
alt_agg_raw = live_equal_support_across_images_df.sort_values("bin_mid_um").copy()
merged_fw_raw = live_trace_merged_df.sort_values("bin_mid_um").copy()
merged_eq_raw = live_equal_support_df.sort_values("bin_mid_um").copy()

LIVE_TRACE_COLOR = "#c62828"
LIVE_TRACE_EDGE_COLOR = "#8e0000"
LIVE_TRACE_XMIN = 0.0
LIVE_TRACE_EQUAL_SUPPORT_XMIN = 50.0
LIVE_TRACE_XMAX = 700.0
LIVE_TRACE_ACROSS_YLIM = (0.0, 1.6)
LIVE_TRACE_MERGED_YLIM = (-0.5, 2.25)
LIVE_CANDIDATE_EQUAL_SUPPORT_PNG = CANDIDATE_FIGURE_DIR / "live_tagyfp_distance_plot_equal_support.png"
LIVE_ALTERNATE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "live_tagyfp_distance_plot_fixed_width.png"
LIVE_SUPPORT_PNG = ALTERNATE_FIGURE_DIR / "live_tagyfp_support_fixed_width.png"
LIVE_MERGED_EQUAL_SUPPORT_PNG = ALTERNATE_FIGURE_DIR / "live_tagyfp_all_pixels_merged_equal_support.png"
LIVE_MERGED_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "live_tagyfp_all_pixels_merged_fixed_width.png"


def _peak_within_window(df: pd.DataFrame, mean_col: str, xmin: float) -> float:
    sub = df[
        np.isfinite(df["bin_mid_um"].astype(float))
        & np.isfinite(df[mean_col].astype(float))
        & (df["bin_mid_um"].astype(float) >= xmin)
        & (df["bin_mid_um"].astype(float) <= LIVE_TRACE_XMAX)
    ].copy()
    vals = sub[mean_col].to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    peak = float(np.nanmax(vals)) if vals.size else 1.0
    return peak if np.isfinite(peak) and peak > 1e-8 else 1.0


def _normalize_live_plot_df(df: pd.DataFrame, scale: float) -> pd.DataFrame:
    out = df.copy()
    for col in [
        "mean_value",
        "median_value",
        "std_value",
        "sem_value",
        "mean",
        "sd",
        "sem",
        "weighted_mean_value",
        "pooled_sd_value",
        "pooled_sem_value",
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype(float) / float(scale)
    return out


main_scale = _peak_within_window(main_agg_raw, "mean", LIVE_TRACE_XMIN)
alt_scale = _peak_within_window(alt_agg_raw, "mean", LIVE_TRACE_EQUAL_SUPPORT_XMIN)
merged_fw_scale = _peak_within_window(merged_fw_raw, "weighted_mean_value", LIVE_TRACE_XMIN)
merged_eq_scale = _peak_within_window(merged_eq_raw, "mean_value", LIVE_TRACE_EQUAL_SUPPORT_XMIN)

main_per_plot = _normalize_live_plot_df(live_stats_df, main_scale)
alt_per_plot = _normalize_live_plot_df(live_stats_df, alt_scale)
main_agg = _normalize_live_plot_df(main_agg_raw, main_scale)
alt_agg = _normalize_live_plot_df(alt_agg_raw, alt_scale)
merged_fw = _normalize_live_plot_df(merged_fw_raw, merged_fw_scale)
merged_eq = _normalize_live_plot_df(merged_eq_raw, merged_eq_scale)

support_sub = merged_fw_raw[
    (merged_fw_raw["bin_mid_um"].astype(float) >= LIVE_TRACE_XMIN)
    & (merged_fw_raw["bin_mid_um"].astype(float) <= LIVE_TRACE_XMAX)
].copy()
support_ymax = float(np.ceil(1.03 * float(np.nanmax(support_sub["total_count_px"].astype(float))) / 100000.0) * 100000.0) if len(support_sub) else 1.0

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
_plot_live_distance_trace(
    ax=ax,
    per_image_df=alt_per_plot,
    agg_df=alt_agg,
    x_col="bin_mid_um",
    y_col="mean",
    band_col="sd",
    title="Live FOXF1-YFP reporter activity by nearest-bead distance | equal-support bins (peak-normalized)",
    color=LIVE_TRACE_COLOR,
    band_label="Mean ± SD across images",
)
ax.set_xlim(LIVE_TRACE_EQUAL_SUPPORT_XMIN, LIVE_TRACE_XMAX)
ax.set_ylim(*LIVE_TRACE_ACROSS_YLIM)
plt.tight_layout()
def _save_plot_all_formats(fig, png_path: Path, dpi: int = 180, bbox_inches: str = "tight") -> None:
    png_path = Path(png_path)
    fig.savefig(png_path, dpi=dpi, bbox_inches=bbox_inches)
    fig.savefig(png_path.with_suffix(".pdf"), bbox_inches=bbox_inches)
    fig.savefig(png_path.with_suffix(".svg"), bbox_inches=bbox_inches)

_save_plot_all_formats(fig, LIVE_CANDIDATE_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
_plot_live_distance_trace(
    ax=ax,
    per_image_df=main_per_plot,
    agg_df=main_agg,
    x_col="bin_mid_um",
    y_col="mean",
    band_col="sd",
    title="Live FOXF1-YFP reporter activity by nearest-bead distance | fixed-width bins (peak-normalized)",
    color=LIVE_TRACE_COLOR,
    band_label="Mean ± SD across images",
)
ax.set_xlim(LIVE_TRACE_XMIN, LIVE_TRACE_XMAX)
ax.set_ylim(*LIVE_TRACE_ACROSS_YLIM)
plt.tight_layout()
_save_plot_all_formats(fig, LIVE_ALTERNATE_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.4))
support = merged_fw_raw.sort_values("bin_mid_um")
bar_widths = (support["bin_end_um"] - support["bin_start_um"]).astype(float).to_numpy()
ax.bar(
    support["bin_mid_um"],
    support["total_count_px"],
    width=bar_widths,
    color=LIVE_TRACE_COLOR,
    alpha=0.65,
    edgecolor=LIVE_TRACE_EDGE_COLOR,
    linewidth=0.6,
    align="center",
)
ax.set_title("Transferred-mask support | total retained pixels pooled across images, fixed-width distance bins")
ax.set_xlabel("Distance from nearest bead (um)")
ax.set_ylabel("Total retained pixels in bin")
ax.set_xlim(LIVE_TRACE_XMIN, LIVE_TRACE_XMAX)
ax.set_ylim(0.0, support_ymax)
plt.tight_layout()
_save_plot_all_formats(fig, LIVE_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
eq = merged_eq.sort_values("bin_mid_um")
eq_mean = eq["mean_value"].to_numpy(dtype=float)
eq_sd = eq["std_value"].to_numpy(dtype=float)
ax.plot(eq["bin_mid_um"], eq_mean, color=LIVE_TRACE_COLOR, lw=2.2, marker="o", ms=3, zorder=3, label="Mean trace")
ax.fill_between(
    eq["bin_mid_um"],
    eq_mean - eq_sd,
    eq_mean + eq_sd,
    color=LIVE_TRACE_COLOR,
    alpha=0.18,
    zorder=2,
    label="±SD",
)
ax.set_title("Live FOXF1-YFP reporter activity by nearest-bead distance | all masked pixels merged, equal-support bins (peak-normalized)")
ax.set_xlabel("Mean distance within equal-support bin (um)")
ax.set_ylabel("Normalized mean FOXF1-YFP reporter activity (live)\n(flat-field corrected, per-image bg-subtracted; aggregate peak bin = 1)")
ax.legend(loc="upper left", frameon=False)
ax.set_xlim(LIVE_TRACE_EQUAL_SUPPORT_XMIN, LIVE_TRACE_XMAX)
ax.set_ylim(*LIVE_TRACE_MERGED_YLIM)
ax.text(
    0.98,
    0.95,
    f"Target pixels/bin = {equal_support_target_px:,}\nBins = {len(eq):,}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8,
    bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.9},
)
plt.tight_layout()
_save_plot_all_formats(fig, LIVE_MERGED_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
merged = merged_fw.sort_values("bin_mid_um")
merged_mean = merged["weighted_mean_value"].to_numpy(dtype=float)
merged_sd = merged["pooled_sd_value"].to_numpy(dtype=float)
ax.plot(merged["bin_mid_um"], merged_mean, color=LIVE_TRACE_COLOR, lw=2.2, marker="o", ms=3, zorder=3, label="Mean trace")
ax.fill_between(
    merged["bin_mid_um"],
    merged_mean - merged_sd,
    merged_mean + merged_sd,
    color=LIVE_TRACE_COLOR,
    alpha=0.18,
    zorder=2,
    label="±SD",
)
ax.set_title("Live FOXF1-YFP reporter activity by nearest-bead distance | all masked pixels merged, fixed-width bins (peak-normalized)")
ax.set_xlabel("Distance from nearest bead (um)")
ax.set_ylabel("Normalized mean FOXF1-YFP reporter activity (live)\n(flat-field corrected, per-image bg-subtracted; aggregate peak bin = 1)")
ax.legend(loc="upper left", frameon=False)
ax.set_xlim(LIVE_TRACE_XMIN, LIVE_TRACE_XMAX)
ax.set_ylim(*LIVE_TRACE_MERGED_YLIM)
ax.text(
    0.98,
    0.95,
    f"Median pixels/bin = {equal_support_target_px:,}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8,
    bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.9},
)
plt.tight_layout()
_save_plot_all_formats(fig, LIVE_MERGED_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()

print("Equal-support target pixels/bin:", f"{equal_support_target_px:,}")
print("All pooled masked live pixels used:", f"{len(live_dist_all_um):,}")
print("Fixed-width merged bins:", len(live_trace_merged_df))
print("Equal-support bins:", len(live_equal_support_df))
print()
print("Across-image fixed-width trace head:")
display(live_trace_df.head())
print("All-pixels merged trace head:")
display(live_trace_merged_df.head())
print("Equal-support pooled-pixel trace head:")
display(live_equal_support_df.head())
print("Equal-support across-image trace head:")
display(live_equal_support_across_images_df.head())
